# BTC preprocessing and data-quality review

Run the cells from top to bottom in the project folder. The notebook reads the local `.env` key through `crypto_matter.py`; it never prints the key. It fetches CryptoMatter's `Close`, `Market_cap`, and `Trade_volume` histories, prepares daily features, displays the missing and invalid observations, and saves `data/btc_daily_preprocessed.csv`.

**Policy:** No forward filling. A missing or nonpositive close stays blank in the cleaned `close` column. Returns require a valid close on both consecutive calendar days.

In [1]:
from pathlib import Path
import math

import numpy as np
import pandas as pd

from crypto_matter import get_series_as_rows

PROJECT_DIR = Path.cwd()
assert (PROJECT_DIR / "crypto_matter.py").exists(), "Open the notebook from the CM project folder."
OUTPUT = PROJECT_DIR / "data" / "btc_daily_preprocessed.csv"
COLUMNS = [
    "date", "close_raw", "close", "market_cap", "trade_volume",
    "has_close", "invalid_close", "has_market_cap", "has_trade_volume",
    "zero_trade_volume", "price_change", "simple_return", "log_return",
    "volatility_30d_annualized", "drawdown_from_30d_peak",
]
print("Output:", OUTPUT)

Output: D:\FS_Courses\COOP\CM\data\btc_daily_preprocessed.csv


## 1. Fetch and align the three API series

The API returns one complete history per field. A full daily calendar makes absent dates explicit.

In [2]:
series = {}
for column in ("Close", "Market_cap", "Trade_volume"):
    observations = get_series_as_rows("bitcoin", column)
    values = pd.Series(
        [value for day, value in observations],
        index=pd.to_datetime([day for day, value in observations]),
        dtype="float64",
        name=column,
    )
    if values.index.has_duplicates:
        raise ValueError(f"Duplicate dates in {column}")
    if not np.isfinite(values.dropna()).all():
        raise ValueError(f"Non-finite values in {column}")
    if column != "Close" and (values.dropna() < 0).any():
        raise ValueError(f"Negative values in {column}")
    series[column] = values
    print(f"{column}: {len(values):,} API rows, {values.index.min().date()} to {values.index.max().date()}")

raw = pd.concat(series.values(), axis=1).sort_index()
calendar = pd.date_range(raw.index.min(), raw.index.max(), freq="D", name="date")
daily = raw.reindex(calendar).rename(
    columns={"Close": "close_raw", "Market_cap": "market_cap", "Trade_volume": "trade_volume"}
)
print(f"Calendar: {len(daily):,} days, {calendar.min().date()} to {calendar.max().date()}")

Close: 4,796 API rows, 2013-04-29 to 2026-10-31
Market_cap: 4,796 API rows, 2013-04-29 to 2026-10-31
Trade_volume: 4,796 API rows, 2013-04-29 to 2026-10-31
Calendar: 4,934 days, 2013-04-29 to 2026-10-31


## 2. Clean prices and calculate features

`close_raw` preserves the API value. `close` excludes missing and nonpositive prices. The rolling features require complete calendar windows: 30 valid daily returns for volatility and 30 valid closes for drawdown.

In [3]:
daily["invalid_close"] = (daily["close_raw"].notna() & daily["close_raw"].le(0)).astype(int)
daily["close"] = daily["close_raw"].where(daily["close_raw"].gt(0))
daily["has_close"] = daily["close"].notna().astype(int)
daily["has_market_cap"] = daily["market_cap"].notna().astype(int)
daily["has_trade_volume"] = daily["trade_volume"].notna().astype(int)
daily["zero_trade_volume"] = daily["trade_volume"].eq(0).astype(int)

previous_close = daily["close"].shift(1)
daily["price_change"] = daily["close"] - previous_close
daily["simple_return"] = daily["close"] / previous_close - 1
daily["log_return"] = np.log(daily["close"] / previous_close)
daily["volatility_30d_annualized"] = (
    daily["log_return"].rolling(30, min_periods=30).std(ddof=1) * math.sqrt(365)
)
daily["drawdown_from_30d_peak"] = (
    daily["close"] / daily["close"].rolling(30, min_periods=30).max() - 1
)
daily.index.name = "date"
prepared = daily.reset_index()[COLUMNS]
print(prepared.tail(5).to_string(index=False))

      date  close_raw   close   market_cap  trade_volume  has_close  invalid_close  has_market_cap  has_trade_volume  zero_trade_volume  price_change  simple_return  log_return  volatility_30d_annualized  drawdown_from_30d_peak
2026-10-27    80814.0 80814.0 1.624750e+12  2.423050e+10          1              0               1                 1                  0         796.0       0.009948    0.009899                        NaN                     NaN
2026-10-28    81274.4 81274.4 1.634020e+12  2.593930e+10          1              0               1                 1                  0         460.4       0.005697    0.005681                        NaN                     NaN
2026-10-29    77878.0 77878.0 1.565570e+12  4.071860e+10          1              0               1                 1                  0       -3396.4      -0.041789   -0.042688                        NaN                     NaN
2026-10-30    77221.0 77221.0 1.552560e+12  2.017580e+11          1              0      

## 3. See the missing and invalid BTC closes

An **absent API close** has no `close_raw` value. An **invalid API close** has a value of zero or less. The two counts are separate; both result in a blank cleaned close. `zero_trade_volume` is a third, separate quality flag.

In [4]:
absent = prepared.loc[prepared["close_raw"].isna(), ["date", "close_raw", "close", "log_return"]]
invalid = prepared.loc[prepared["invalid_close"].eq(1), ["date", "close_raw", "close", "log_return"]]
summary = pd.DataFrame({
    "check": [
        "Calendar days",
        "Absent API closes",
        "Invalid nonpositive API closes",
        "Cleaned closes unavailable",
        "Zero trade-volume values",
        "Usable daily log returns",
        "Available 30-day volatility values",
    ],
    "count": [
        len(prepared),
        len(absent),
        len(invalid),
        prepared["close"].isna().sum(),
        prepared["zero_trade_volume"].sum(),
        prepared["log_return"].notna().sum(),
        prepared["volatility_30d_annualized"].notna().sum(),
    ],
})
print(summary.to_string(index=False))
print("\nSix invalid-price rows (if the API history is unchanged):")
print(invalid.to_string(index=False))
print("\nMost recent 15 dates absent from the API:")
print(absent.tail(15).to_string(index=False))

                             check  count
                     Calendar days   4934
                 Absent API closes    138
    Invalid nonpositive API closes      6
        Cleaned closes unavailable    144
          Zero trade-volume values    239
          Usable daily log returns   4649
Available 30-day volatility values   3450

Six invalid-price rows (if the API history is unchanged):
      date  close_raw  close  log_return
2014-12-31        0.0    NaN         NaN
2016-02-29        0.0    NaN         NaN
2016-11-30        0.0    NaN         NaN
2018-10-31        0.0    NaN         NaN
2018-12-31        0.0    NaN         NaN
2020-02-29        0.0    NaN         NaN

Most recent 15 dates absent from the API:
      date  close_raw  close  log_return
2020-02-15        NaN    NaN         NaN
2020-02-18        NaN    NaN         NaN
2020-03-11        NaN    NaN         NaN
2020-03-14        NaN    NaN         NaN
2020-03-25        NaN    NaN         NaN
2020-11-25        NaN    NaN 

## 4. Inspect a zero-price day and its neighboring returns

The raw zero remains visible. The cleaned price and that day's return are blank; the next day's return is also blank because its previous close is invalid.

In [5]:
example = pd.Timestamp("2014-12-31")
window = prepared.loc[
    prepared["date"].between(example - pd.Timedelta(days=2), example + pd.Timedelta(days=2)),
    ["date", "close_raw", "close", "invalid_close", "price_change", "simple_return", "log_return"],
]
print(window.to_string(index=False))
print("\nLatest 30-day volatility observation:")
available_vol = prepared.loc[prepared["volatility_30d_annualized"].notna(), "date"]
print(available_vol.iloc[-1].date() if not available_vol.empty else "none")

      date  close_raw   close  invalid_close  price_change  simple_return  log_return
2014-12-29    312.633 312.633              0           NaN            NaN         NaN
2014-12-30    317.379 317.379              0         4.746       0.015181    0.015067
2014-12-31      0.000     NaN              1           NaN            NaN         NaN
2015-01-01    313.992 313.992              0           NaN            NaN         NaN
2015-01-02    314.446 314.446              0         0.454       0.001446    0.001445

Latest 30-day volatility observation:
2026-10-12


## 5. Save and verify the CSV

Blank numeric fields in the CSV mean unavailable values, not zero. Rerunning this cell updates the CSV. If the existing CSV is open and Windows prevents replacement, the notebook saves `btc_daily_preprocessed_notebook.csv` instead.

In [6]:
OUTPUT.parent.mkdir(exist_ok=True)
temporary = OUTPUT.with_suffix(".csv.tmp")
prepared.to_csv(temporary, index=False, date_format="%Y-%m-%d")
try:
    temporary.replace(OUTPUT)
except PermissionError:
    OUTPUT = OUTPUT.with_name("btc_daily_preprocessed_notebook.csv")
    temporary.replace(OUTPUT)
    print("Original CSV was locked; saved a separate notebook copy.")

check = pd.read_csv(OUTPUT, parse_dates=["date"])
assert len(check) == len(prepared)
assert check["date"].is_unique
assert check["date"].is_monotonic_increasing
assert int(check["close_raw"].isna().sum()) == len(absent)
assert int(check["invalid_close"].sum()) == len(invalid)
assert check.loc[check["close"].isna(), "log_return"].isna().all()
print(f"Saved and verified {len(check):,} rows: {OUTPUT}")

Saved and verified 4,934 rows: D:\FS_Courses\COOP\CM\data\btc_daily_preprocessed.csv
